# 🧠 Notebook 07: Trap Taxonomy and Failure

## 1. Purpose + Scope

This notebook categorizes the various failure modes (traps) in the T81 VM:

*   **Deterministic Traps**: Exceptions that must occur at the same instruction on all nodes (e.g., division by zero).
*   **OOM_QUOTA Simulation**: Running out of allocated memory.
*   **Strict Violations**: Attempting forbidden operations in strict mode.
*   **Replay Invariants**: Ensuring failures are reproducible.

## 2. Spec References

*   `spec/t81vm-spec.md`
*   `include/t81/vm/vm.hpp`

## 3. Determinism Tier

**Tier A (Strict Determinism)**: A trap is a deterministic state transition. The VM halts with a specific error code, and this halt must be identical across all replicas.

## 4. Reproducibility Setup

Ensure `t81_python` is built and available in `PYTHONPATH`.

In [ ]:
import sys
import os

build_dir = os.path.abspath(os.path.join(os.getcwd(), "../build"))
if build_dir not in sys.path:
    sys.path.append(build_dir)

try:
    import t81_python
    print("✅ t81_python loaded.")
except ImportError:
    print("❌ Failed to load t81_python.")
    sys.exit(1)

## 5. Exploratory Code: Division by Zero Trap

A classic deterministic trap.

In [ ]:
div_zero_src = """
fn main() -> T81BigInt {
    let a: T81BigInt = 10t81;
    let b: T81BigInt = 0t81;
    return a / b;
}
"""

try:
    prog = t81_python.compile(div_zero_src)
    vm = t81_python.make_interpreter_vm()
    vm.load_program(prog)
    print("Running division by zero...")
    vm.run_to_halt()
except RuntimeError as e:
    print(f"✅ Caught expected trap: {e}")

## 6. OOM_QUOTA Simulation

If the VM enforces a memory limit, allocating too much should trap.

In [ ]:
# Simulating memory pressure via recursion or large allocations if supported by bindings.
# Here we use infinite recursion to trigger stack overflow or step limit.
recursion_src = """
fn recurse(n: T81BigInt) -> T81BigInt {
    return recurse(n + 1t81);
}
fn main() -> T81BigInt {
    return recurse(0t81);
}
"""

try:
    prog = t81_python.compile(recursion_src)
    vm = t81_python.make_interpreter_vm()
    vm.load_program(prog)
    print("Running infinite recursion...")
    vm.run_to_halt(max_steps=5000)
except RuntimeError as e:
    print(f"✅ Caught expected trap (Stack Overflow / Limit): {e}")

## 7. Strict Mode Violations

Certain operations might be valid in relaxed mode but trap in strict mode (e.g., non-deterministic float ops). This requires configuring the VM mode, which we simulate here or set if bindings expose it.

## 8. Architectural Commentary

Traps are not crashes. They are valid termination states. The consensus engine must agree that a transaction failed with a specific trap code.